# 第 1 節｜20 分鐘實作：郵件分類與模型評估

分類垃圾郵件（1），再觀察分類錯誤、決策樹和 SVM。

In [ ]:
# Load Pandas to work with email tables.
# 載入 Pandas，處理郵件表格。
import pandas as pd

# Load NumPy for a simple comparison baseline.
# 載入 NumPy，建立簡單的比較基準。
import numpy as np

# Load Matplotlib for classroom charts.
# 載入 Matplotlib，繪製教學圖表。
import matplotlib.pyplot as plt

# Find CSV files in Colab or local folders.
# 使用 Path 尋找 Colab 或本機資料集。
from pathlib import Path

# Split training and test data.
# 匯入訓練與測試集切分工具。
from sklearn.model_selection import train_test_split

# Train a simple decision-tree classifier.
# 匯入簡單的決策樹分類模型。
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Compare linear and nonlinear SVM models.
# 匯入線性及非線性 SVM 模型。
from sklearn.svm import SVC

# Import all classification evaluation metrics.
# 匯入分類指標與混淆矩陣視覺化工具。
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay

## 1. 讀取資料

`is_spam`：1 是垃圾郵件，0 是正常郵件。資料為模擬練習資料。

| 欄位名稱 | 中文名稱 | 資料型態 | 說明與取值 | 教學用途 |
|---|---|---|---|---|
| `email_id` | 郵件編號 | 文字 | `E001`～`E500`，每封郵件的唯一編號 | 僅作識別，**不加入 X** |
| `link_count` | 連結數量 | 非負整數 | 郵件中的網址連結數；本資料範圍 0～6 | 數值型特徵 X |
| `urgent_words` | 緊急字詞數量 | 非負整數 | 郵件中「立即」、「限時」、「緊急」等字詞的出現次數；範圍 0～5 | 數值型特徵 X |
| `exclamation_count` | 驚嘆號數量 | 非負整數 | 郵件中 `!` 出現的次數；範圍 0～5 | 數值型特徵 X |
| `sender_known` | 是否為已知寄件者 | 二元類別 | `1`＝已知寄件者；`0`＝未知寄件者 | 0／1 特徵 X |
| `email_type` | 郵件類型 | 名目型類別 | `校務公告`、`社團活動`、`課程通知`、`優惠資訊` | 類別型特徵 X，使用 One-Hot Encoding |
| `is_spam` | 是否為垃圾郵件 | 二元類別 | `1`＝垃圾郵件；`0`＝正常郵件 | **分類目標 y** |

In [ ]:
# Find the CSV in Colab first.
# 優先從 Colab 的 /content 讀取資料。
file_path = Path("/content/02_校園郵件分類_500筆.csv")

# Use the local folder when not running on Colab.
# 如果不是在 Colab，就使用目前工作目錄。
if not file_path.exists():
    # Read from the current notebook folder.
    # 從目前 Notebook 工作目錄讀取。
    file_path = Path("02_校園郵件分類_500筆.csv")

# Load the 500 simulated school emails.
# 讀取 500 筆模擬校園電子郵件資料。
df = pd.read_csv(file_path)

# Preview only the first three email records.
# 只查看前三筆郵件資料，避免輸出太多。
df.head(3)

## 2. 切分訓練／測試資料，再合併數值與類別特徵

`email_type` 使用 One-Hot，其餘數值欄位直接保留。

In [ ]:
# Separate the email features from the correct labels.
# 將郵件特徵與正確答案分開。
X = df.drop(columns=["email_id", "is_spam"])

# Set aside 20 percent as untouched test data.
# 先保留 20% 當測試集，並維持垃圾郵件比例。
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, df["is_spam"], test_size=0.2, stratify=df["is_spam"], random_state=42)

In [ ]:
# Encode the training categories as binary columns.
# 將訓練資料的郵件類型轉成 One-Hot 欄位。
X_train = pd.get_dummies(X_train_raw, dtype=int)

# Match the test columns to the training feature columns.
# 測試資料沿用相同欄位，確保模型輸入一致。
X_test = pd.get_dummies(X_test_raw, dtype=int).reindex(columns=X_train.columns, fill_value=0)

# Preview the numerical and encoded categorical features.
# 檢查數值特徵和類別特徵已合併成 X。
X_train.head(3)

## 3. Accuracy 的陷阱

如果全部都猜正常郵件，Accuracy 可能不低，卻完全抓不到垃圾郵件。

In [ ]:
# Predict every test email as normal.
# 假設所有測試郵件都被預測為正常郵件。
always_normal = np.zeros(len(y_test), dtype=int)

# Check how high the misleading accuracy can be.
# 觀察全部猜正常的 Accuracy。
print("Accuracy:", round(accuracy_score(y_test, always_normal), 2))

# Check whether any actual spam was detected.
# 觀察是否找回任何真正的垃圾郵件。
print("Recall:", round(recall_score(y_test, always_normal), 2))

## 4. Decision Tree：訓練並畫出決策規則

`max_depth=3` 限制樹的深度，避免畫面過於複雜。

In [ ]:
# Build an easy-to-read decision tree.
# 建立最多三層的決策樹。
tree = DecisionTreeClassifier(max_depth=3, random_state=42)

# Learn classification rules from the training emails.
# 讓模型從訓練郵件學習分類規則。
tree.fit(X_train, y_train)

# Prepare a readable figure.
# 建立足夠大的圖表，方便看清楚決策問題。
plt.figure(figsize=(13, 5))

# Draw the first two levels of the learned tree.
# 畫出決策樹前兩層的切分規則。
plot_tree(tree, max_depth=2, feature_names=X_train.columns.str.replace("優惠資訊", "Promotion"), class_names=["Normal", "Spam"], filled=True, fontsize=9)

# Show the decision-tree diagram.
# 顯示決策樹圖表。
plt.show()

## 5. 模型評估：四個指標與混淆矩陣

本課把「垃圾郵件」當正類（1）；重點是分清誤擋 FP 與漏抓 FN。

In [ ]:
# Predict the labels of the held-out test emails.
# 預測測試資料中哪些郵件是垃圾郵件。
pred = tree.predict(X_test)

# Measure the fraction of all correct classifications.
# 計算整體預測正確率 Accuracy。
print("Accuracy:", round(accuracy_score(y_test, pred), 2))

# Measure how reliable spam predictions are.
# 計算預測為垃圾郵件時的 Precision。
print("Precision:", round(precision_score(y_test, pred), 2))

# Measure how many real spam messages were found.
# 計算真正垃圾郵件的 Recall。
print("Recall:", round(recall_score(y_test, pred), 2))

# Balance precision and recall using the F1 score.
# 計算綜合 Precision 與 Recall 的 F1。
print("F1:", round(f1_score(y_test, pred), 2))

In [ ]:
# Draw the real-versus-predicted confusion matrix.
# 繪製真實答案與預測結果的混淆矩陣。
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["Normal", "Spam"], cmap="Blues")

# Show all four confusion-matrix cells.
# 顯示 TN、FP、FN、TP 四格。
plt.show()

## 6. SVM：比較 Linear 與 RBF

兩個模型使用**相同資料與 F1** 比較；RBF 不一定比較好。

In [ ]:
# Create the straight-boundary SVM model.
# 建立線性邊界的 SVM 模型。
linear_svm = SVC(kernel="linear")

# Create the flexible nonlinear SVM model.
# 建立能處理非線性分界的 RBF SVM。
rbf_svm = SVC(kernel="rbf")

# Train the linear SVM on the same features.
# 使用相同特徵訓練 Linear SVM。
linear_svm.fit(X_train, y_train)

# Train the RBF SVM for a fair comparison.
# 使用相同資料訓練 RBF SVM。
rbf_svm.fit(X_train, y_train);

In [ ]:
# Compare the linear SVM using test F1.
# 查看 Linear SVM 的測試 F1。
print("Linear F1:", round(f1_score(y_test, linear_svm.predict(X_test)), 2))

# Compare the nonlinear SVM using test F1.
# 查看 RBF SVM 的測試 F1。
print("RBF F1:", round(f1_score(y_test, rbf_svm.predict(X_test)), 2))

## 7. 新的三封郵件：交給訓練完成的模型

修改新郵件的數字，觀察結果如何改變，不需要重新 `fit()`。

In [ ]:
# Create three new emails without correct-answer labels.
# 建立三筆沒有預先給答案的新郵件。
new_data = pd.DataFrame({
    "link_count": [0, 2, 5],
    "urgent_words": [0, 2, 4],
    "exclamation_count": [0, 2, 4],
    "sender_known": [1, 0, 0],
    "email_type": ["校務公告", "社團活動", "優惠資訊"]
})

# Reuse the exact feature columns learned during training.
# 新郵件必須和訓練資料使用相同的 One-Hot 欄位。
new_X = pd.get_dummies(new_data, dtype=int).reindex(columns=X_train.columns, fill_value=0)

# Predict all three new emails using the trained tree.
# 用訓練完成的決策樹預測三封新郵件。
new_data["預測（1=垃圾）"] = tree.predict(new_X)

# Display the original features and predicted classes.
# 一起顯示新郵件內容與預測結果。
new_data